In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
openrouter_client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()

In [2]:
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

from minsearch import Index
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)

from rag_helper_modified import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openrouter_client
)

# How many lesson pages are in the dataset?
len(documents)

72

In [3]:
question = "How does the agentic loop keep calling the model until it stops?"

In [4]:
search_results = assistant.search(question)

# What's the filename of the first result?
search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [5]:
response = assistant.rag(question)
response

Response(id='gen-1782062437-CGnhO5zk3Sp43xkHXFDc', created_at=1782062438.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='openai/gpt-5.4-mini-20260317', object='response', output=[ResponseOutputMessage(id='msg_tmp_trnnf9yh8d', content=[ResponseOutputText(annotations=[], text='Searching the lesson text for how the loop decides to continue and stop.Searching for the core loop condition and the `has_function_calls` flag.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='commentary'), ResponseOutputMessage(id='msg_tmp_zhr2lks720r', content=[ResponseOutputText(annotations=[], text='The agentic loop keeps calling the model by using a `while True` loop plus a flag like `has_function_calls`.\n\nHow it works:\n\n1. Call the model with the current `messages`.\n2. Check the model’s output.\n3. If it returns a `function_call`, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.\n4. Ca

In [6]:
usage = response.usage

# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?
usage.input_tokens

7118

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

# How many chunks do you get?
len(chunks)

295

In [8]:
index_chunks = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index_chunks.fit(chunks)

assistant_chunks = RAGBase(
    index=index_chunks,
    llm_client=openrouter_client,
)
response_chunks = assistant_chunks.rag(question)
usage_chunks = response_chunks.usage

# How many fewer input tokens does the chunked version send?
usage_chunks.input_tokens

2301

In [11]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> dict[str, str]:
    """
    Search the lessons for entries matching the given query.
    """
    return index_chunks.search(
        query,
        num_results=5
    )
    
agent_tools = Tools()
agent_tools.add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
'''

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="openai/gpt-5.4-mini", client=openrouter_client)
)

question2 = "How does the agentic loop work, and how is it different from plain RAG?"
result = runner.loop(
    prompt=question2,
    callback=callback,
)

# How many times did the agent call search?

-> Response received


-> Response received
